# Tester

> **Reference:** Barrett et al., *arXiv:2502.12870* (2025). GitHub: https://github.com/rhyan10/X-MACE

## 1. Dataset

In [8]:
import ase.io
import numpy as np

train_file = "data/A01_ethene_grid_static_CASSCF.xyz"
db    = ase.io.read(train_file, ":5")
atoms = db[0]

N_atoms  = len(atoms)                            
n_energies = 3
n_pairs  = n_states * (n_states - 1) // 2        

print(f"Atoms per frame : {N_atoms}")
print(f"Electronic states: {n_energies}")
print(f"State pairs (NAC): {n_pairs}")
print()

# Quick shape summary
keys_to_check = {
    'REF_energy'      : (1, n_energies),
    'REF_forces'      : (N_atoms, n_energies, 3),
}
for key, expected in keys_to_check.items():
    actual = np.array(atoms.info[key]).shape
    status = "✓" if actual == expected else f"✗ got {actual}"
    print(f"  {key:15s}  expected {str(expected):15s}  {status}")

Atoms per frame : 6
Electronic states: 3
State pairs (NAC): 3

  REF_energy       expected (1, 3)           ✓
  REF_forces       expected (6, 3, 3)        ✓


## 2. Train X-MACE (AutoencoderExcitedMACE)

In [11]:
# Run this cell to launch training (requires a GPU and X-MACE installed)

model = "AutoencoderExcitedMACE"
r_max = 5.0
max_num_epochs = 100
lr = 0.0001
energy_weight = 100.0
forces_weight = 100.0

xmace_cmd = f"""
python scripts/run_train.py \
  --name="energies_forces" \
  --train_file="{train_file}" \
  --seed=100 \
  --valid_fraction=0.1 \
  --E0s='average' \
  --model="{model}" \
  --r_max={r_max} \
  --batch_size=10 \
  --n_energies={n_energies} \
  --correlation=3 \
  --max_num_epochs={max_num_epochs} \
  --ema \
  --lr={lr} \
  --ema_decay=0.99 \
  --default_dtype="float32" \
  --device=cuda \
  --hidden_irreps="128x0e + 128x1o" \
  --MLP_irreps='128x0e' \
  --num_radial_basis=8 \
  --num_interactions=2 \
  --energy_weight={energy_weight} \
  --forces_weight={forces_weight} \
  --error_table="EnergyNacsDipoleMAE"
"""

import torch
print("Is CUDA available?:", torch.cuda.is_available())
print("Torch version:", torch.__version__)

import subprocess
subprocess.run(xmace_cmd, shell=True, check=True)

/home/yutong/micromamba/envs/x-mace-env/lib/python3.13/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Is CUDA available?: False
Torch version: 2.7.1+cu118


ERROR:root:No token file found. Also make sure that a [prod] section with a 'token = value' assignment exists.
INFO:root:===========VERIFYING SETTINGS===========
INFO:root:MACE version: 0.3.6
DEBUG:root:Configuration: Namespace(name='energies_forces', seed=100, work_dir='.', nacs_key='smooth_nacs', log_dir='./logs', model_dir='.', checkpoints_dir='./checkpoints', results_dir='./results', downloads_dir='./downloads', device='cuda', default_dtype='float32', distributed=False, log_level='INFO', n_energies=3, error_table='EnergyNacsDipoleMAE', model='AutoencoderExcitedMACE', r_max=5.0, num_permutational_invariant=16, radial_type='bessel', num_radial_basis=8, num_cutoff_basis=5, pair_repulsion=False, distance_transform='None', interaction='RealAgnosticResidualInteractionBlock', interaction_first='RealAgnosticResidualInteractionBlock', max_ell=3, correlation=3, num_interactions=2, MLP_irreps='128x0e', radial_MLP='[64, 64, 64]', hidden_irreps='128x0e + 128x1o', num_channels=128, max_L=1, gate

2026-07-02 12:09:08.728 INFO: ===========VERIFYING SETTINGS===========
2026-07-02 12:09:08.728 INFO: MACE version: 0.3.6


CalledProcessError: Command '
python scripts/run_train.py   --name="energies_forces"   --train_file="data/A01_ethene_grid_static_CASSCF.xyz"   --seed=100   --valid_fraction=0.1   --E0s='average'   --model="AutoencoderExcitedMACE"   --r_max=5.0   --batch_size=10   --n_energies=3   --correlation=3   --max_num_epochs=100   --ema   --lr=0.0001   --ema_decay=0.99   --default_dtype="float32"   --device=cuda   --hidden_irreps="128x0e + 128x1o"   --MLP_irreps='128x0e'   --num_radial_basis=8   --num_interactions=2   --energy_weight=100.0   --forces_weight=100.0   --error_table="EnergyNacsDipoleMAE"
' returned non-zero exit status 1.